In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Setup & Installations

In [1]:
!pip install -q git+https://github.com/huggingface/transformers.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 11.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 81.7 MB/s eta 0:00:00:00:01


In [2]:
!pip install -q accelerate bitsandbytes sentencepiece langchain_core langchain_huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 31.2 MB/s eta 0:00:00


In [3]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secret_label = "HF_FACE"
secret_value = UserSecretsClient().get_secret(secret_label)
 
login(secret_value)

# Imports 

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import Optional, List
import json


In [5]:
MODEL_NAME="google/gemma-2-9b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config, # to prevent OOM
    device_map="auto"               
)

config.json:   0%|          | 0.00/857 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

In [49]:
pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.1,
    do_sample=True,
    repetition_penalty=1.2,
    return_full_text=False
)

llm = HuggingFacePipeline(pipeline=pipeline)

ValueError: The following `model_kwargs` are not used by the model: ['model'] (note: typos in the generate arguments will also show up in this list)

# Agent1 Data Extractor

### Data Schema

In [15]:
class TestScores(BaseModel):
    """Schema for standardized test scores"""
    ielts: Optional[float] = Field(None, description="IELTS band score (0-9)")
    toefl: Optional[int] = Field(None, description="TOEFL score (0-120)")
    duolingo: Optional[int] = Field(None, description="Duolingo score (10-160)")
    gre_verbal: Optional[int] = Field(None, description="GRE Verbal Reasoning score")
    gre_quant: Optional[int] = Field(None, description="GRE Quantitative Reasoning score")
    gre_awa: Optional[float] = Field(None, description="GRE Analytical Writing score")


class UserProfile(BaseModel):
    """The final cleaned profile of the scholarship applicant"""
    university: Optional[str] = Field(None, description="University name")
    degree_and_specialty: Optional[str] = Field(None, description="Degree and major, e.g., BSc Computer Science")
    gpa: Optional[float] = Field(None, description="Grade Point Average")
    gpa_scale: Optional[float] = Field(4.0, description="GPA scale, e.g., 4.0 or 5.0")
    test_scores: TestScores = Field(default_factory=TestScores)
    project_titles: List[str] = Field(default_factory=list, description="List of project titles")
    published_research_titles: List[str] = Field(default_factory=list, description="List of published research paper titles")
    competition_wins: List[str] = Field(default_factory=list, description="List of competitions won")
    volunteering_activities: List[str] = Field(default_factory=list, description="List of volunteering activities")

In [24]:
parser = PydanticOutputParser(pydantic_object=UserProfile)

SYSTEM_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """
    
    You are an expert Admissions Profiler AI. Your job is to extract specific information from a user's resume or profile text and format it as JSON.

Extraction Rules:
1. Only extract information explicitly stated in the text. Do NOT guess or hallucinate.
2. If a field is not mentioned, return null for strings/numbers, and an empty array [] for lists.
3. For GPA, also try to identify the scale (e.g., 4.0 or 5.0). If not stated, assume 4.0.
4. Extract the EXACT titles of projects, research papers, competitions, and volunteering. Do not summarize them.

{format_instructions}

Roles: 
Respond ONLY with valid JSON.
Do not include explanations or markdown.

"""),
    ("human", "Here is the applicant's data:\n\n{applicant_text}")
])

print("-- Model loaded in 4-bit and LangChain chain created successfully! --")

✅ Model loaded in 4-bit and LangChain chain created successfully!


In [43]:
agent1_full_chain = (
    SYSTEM_PROMPT.partial(
        format_instructions=parser.get_format_instructions()
    )
    | llm)

# Test

In [45]:
sample_text = """

Hi, my name is Abdallah. 
I graduated from the University of Toronto with a BSc in Computer Science.
My GPA is 3.8 out of 4.0. 
I took the TOEFL and scored 112. 
Also did the GRE and got Verbal 162, Quant 170, AWA 4.5.
For projects, I built a "Real-time Sign Language Translator" and a "Decentralized Voting System using Blockchain". 
I have two published papers: "Optimizing LSTM for Real-Time Video Processing" and "Blockchain Consensus Mechanisms in IoT Networks". 
I won 1st place in the Google Hackathon 2023 and got a Bronze medal in the ICPC Regional Finals. 
In my free time, I volunteer at Code.org teaching kids to code, and I also help out at the local animal shelter.

"""

print("\nRunning Agent 1")
try:
    profile = chain.invoke({"applicant_text": sample_text})
    if hasattr(profile, "content"):
        profile = profile.content

    cleaned = extract_json(profile)


    data = json.loads(cleaned)

    profile = UserProfile(**data)

    
    print("\n-- EXTRACTION SUCCESSFUL! --")
    print("-" * 30)
    print(f"University:          {profile.university}")
    print(f"Degree:              {profile.degree_and_specialty}")
    print(f"GPA:                 {profile.gpa}/{profile.gpa_scale}")
    print(f"TOEFL:               {profile.test_scores.toefl}")
    print(f"GRE (V/Q/AWA):       {profile.test_scores.gre_verbal} / {profile.test_scores.gre_quant} / {profile.test_scores.gre_awa}")
    print(f"Projects:            {profile.project_titles}")
    print(f"Research Titles:     {profile.published_research_titles}")
    print(f"Competitions:        {profile.competition_wins}")
    print(f"Volunteering:        {profile.volunteering_activities}")
    
except Exception as e:
    print(f"\n-- EXTRACTION FAILED: {e} --")

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Running Agent 1: Admissions Profiler...

EXTRACTION SUCCESSFUL!
------------------------------
University:          University of Toronto
Degree:              BSc in Computer Science
GPA:                 3.8/4.0
TOEFL:               112
GRE (V/Q/AWA):       162 / 170 / 4.5
Projects:            ['Real-time Sign Language Translator', 'Decentralized Voting System using Blockchain']
Research Titles:     ['Optimizing LSTM for Real-Time Video Processing', 'Blockchain Consensus Mechanisms in IoT Networks']
Competitions:        ['Google Hackathon 2023 - 1st Place', 'ICPC Regional Finals - Bronze Medal']
Volunteering:        ['Code.org - Teaching kids to code', 'Local Animal Shelter']
